# BME i9400 — Meeting 3
## Studio: Screening Simulation

**Fall 2026 · Wednesday, September 9**

Today's studio has two purposes: an introduction to NumPy and pandas, and a simulation of the
screening problem from Meeting 2. The screening problem serves as the worked
example throughout, so the syntax is introduced against a problem you have already solved
analytically.

**Agenda**

1. NumPy: arrays, random numbers, boolean masks
2. pandas: dataframes and cross-tabulation
3. Simulating a screened population
4. Counting the four outcomes
5. Recovering sensitivity, specificity, PPV, and NPV
6. Sweeping prevalence
7. Submitting your work

Sections 1 and 2 are done for you — just read and run them. Sections 3 to 6 contain tasks marked
`TODO` that you are asked to complete. Each task ends with checks that must pass before you move on.

---
## Recap

From Meeting 2, screening mammography in the United States:

| Quantity | Symbol | Value |
|---|---|---|
| Sensitivity | $P(T{=}1 \mid D{=}1)$ | 0.869 |
| Specificity | $P(T{=}0 \mid D{=}0)$ | 0.889 |
| Prevalence | $P(D{=}1)$ | 0.006 |

We computed $P(D{=}1 \mid T{=}1) = 0.045$ analytically. Today we obtain the same number by
simulating a population and counting, which is a check on the algebra and a template for problems
where the algebra is not available.

---
## 1 — NumPy: arrays, random numbers, boolean masks

NumPy provides the array: a fixed-size block of numbers of one type, on which arithmetic is applied
to every element at once.

> An **array** is like a Python list, but every element has the same type and operations apply to the
> whole thing at once. `a * 2` doubles every element without a loop.
>
> **Vectorised** means "applied to the whole array at once, without writing a `for` loop". NumPy code
> is written this way throughout.

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])

print("a          =", a)
print("a * 2      =", a * 2)          # every element, no loop
print("a + a      =", a + a)
print("a.shape    =", a.shape)        # (5,) means one dimension of length 5
print("a.dtype    =", a.dtype)        # the shared element type
print("a.sum()    =", a.sum())
print("a.mean()   =", a.mean())

### Random numbers

Simulation needs random numbers, and reproducible work needs the *same* random numbers every time.
NumPy provides a generator that you seed with a fixed integer.

> A **seed** is a starting value for the random number generator. The same seed produces the same
> sequence of "random" numbers every run. This is what makes a simulation reproducible: your
> classmate running your notebook gets your numbers.

`rng.random(n)` returns `n` numbers drawn uniformly between 0 and 1.

In [ ]:
rng = np.random.default_rng(9400)     # 9400 is the seed here, but any integer works

u = rng.random(10)
print("ten uniform draws:")
print(np.round(u, 3))

print("\nrun it again with the same seed:")
print(np.round(np.random.default_rng(9400).random(10), 3))   # identical

### Boolean masks

Comparing an array to a number gives an array of `True`/`False` — one per element. The simulation
in sections 3 to 6 relies on such comparisons.

> A **boolean mask** is an array of `True`/`False` values, usually the result of a comparison.
>
> In arithmetic, `True` counts as 1 and `False` counts as 0. So `mask.sum()` counts how many
> elements are `True`, and `mask.mean()` gives the *fraction* that are `True`.

`&` means "and", `|` means "or". Each side must be wrapped in parentheses.

In [ ]:
u = rng.random(10)
small = u < 0.5

print("u            =", np.round(u, 3))
print("u < 0.5      =", small)
print("small.sum()  =", small.sum(), " (how many are True)")
print("small.mean() =", small.mean(), " (what fraction are True)")

# Drawing an event with probability p: draw a uniform number and ask if it fell below p.
p = 0.006
draws = rng.random(100_000) < p
print(f"\nfraction below {p}: {draws.mean():.5f}   (should be close to {p})")

# Combining masks with & (and)
big_and_even_index = (u > 0.5) & (np.arange(10) % 2 == 0)
print("\n(u > 0.5) & (even index) =", big_and_even_index)

print("Make sure you understand the output above ^^ .")

---
## 2 — pandas: dataframes and cross-tabulation

pandas provides the dataframe: a table with named columns, where each column is an array.

> A **dataframe** is a table. Each column has a name and holds one variable; each row is one
> observation — here, one woman.

`pd.crosstab` counts how many rows fall into each combination of two columns. That is exactly the
table of four outcomes from Meeting 2.

In [ ]:
import pandas as pd

demo = pd.DataFrame({
    "disease": [True,  True,  False, False, False],
    "test":    [True,  False, True,  False, False],
})

print("Our sample dataframe:")
print(demo, "\n")
print("shape:", demo.shape, "  (rows, columns)\n")
print("Result of pd.crosstab:")
print(pd.crosstab(demo["disease"], demo["test"]))

---
## 3 — Simulating a screened population

Two steps, in order:

1. Decide who has the disease. Each woman independently has the disease with probability equal to
   the prevalence.
2. Decide each woman's test result. This probability **depends on her disease status**:
    - if she has the disease, the test is positive with probability equal to the sensitivity
    - if she does not, the test is positive with probability equal to $1 -$ specificity

Step 2 is where the conditional probability from Meeting 2 enters the code. The probability of a
positive test is not one number; it is one number for the diseased and a different number for the
healthy.

`np.where(condition, x, y)` builds an array by choosing `x` where the condition is `True` and `y`
where it is `False` — which is how we assign a different probability to each group.

In [ ]:
SENS = 0.869              # sensitivity
SPEC = 0.889              # specificity
PREV = 0.006              # prevalence of breast cancer
N    = 200_000            # total number of women screened

rng = np.random.default_rng(9400)

# --- done for you: step 1, who has the disease ---
disease = rng.random(N) < PREV
print(f"simulated {N:,} women; {disease.sum():,} have the disease "
      f"({100*disease.mean():.3f}%, target {100*PREV:.3f}%)")

# --- TODO: step 2, each woman's probability of a positive test ---
# Use np.where to build an array of length N holding SENS for the diseased
# and (1 - SPEC) for everyone else.
p_positive = None      # TODO: replace None

# --- TODO: draw each woman's test result using p_positive ---
test = None            # TODO: replace None

assert p_positive is not None and test is not None, "Fill in both TODOs above."
assert p_positive.shape == (N,), "p_positive should have one entry per woman."
assert np.isclose(p_positive[disease][0], SENS), "Diseased women should get SENS."
assert np.isclose(p_positive[~disease][0], 1 - SPEC), "Healthy women should get 1 - SPEC."
assert test.dtype == bool, "test should be a boolean array."
print(f"{test.sum():,} women tested positive ({100*test.mean():.2f}%)")

---
## 4 — Counting the four outcomes

Each woman falls into exactly one of the four cells from Meeting 2, determined by combining her
`disease` and `test` values.

> `~` means "not". `~disease` is `True` for every woman who does not have the disease.

Build each count with a boolean mask.

In [ ]:
# --- TODO: count each of the four outcomes ---
# Combine the disease and test masks with & and ~, then .sum()
TP = None      # diseased and tested positive
FN = None      # diseased and tested negative
FP = None      # healthy  and tested positive
TN = None      # healthy  and tested negative

assert all(v is not None for v in (TP, FN, FP, TN)), "Fill in all four counts."
assert TP + FN + FP + TN == N, f"The four counts must add to {N:,}; you have {TP+FN+FP+TN:,}."

print(f"{'':<12}{'test +':>12}{'test -':>12}")
print(f"{'diseased':<12}{TP:>12,}{FN:>12,}")
print(f"{'healthy':<12}{FP:>12,}{TN:>12,}")

# Confirm against pandas
df = pd.DataFrame({"disease": disease, "test": test})
print("\npd.crosstab gives the same table:\n")
print(pd.crosstab(df["disease"], df["test"]))

---
## 5 — Recovering sensitivity, specificity, PPV, and NPV

Each of the four quantities is a ratio of counts. The definitions from Meeting 2, written as counts:

$$\text{sensitivity} = \frac{TP}{TP + FN} \qquad \text{specificity} = \frac{TN}{TN + FP}$$

$$\text{PPV} = \frac{TP}{TP + FP} \qquad \text{NPV} = \frac{TN}{TN + FN}$$

The denominators differ. Sensitivity and specificity divide **within a disease row**; PPV and NPV
divide **within a test column**. That is the same distinction as Meeting 2, now visible in the
arithmetic.

In [ ]:
# --- TODO: compute the four quantities from your counts ---
sens_hat = None
spec_hat = None
ppv_hat  = None
npv_hat  = None

assert all(v is not None for v in (sens_hat, spec_hat, ppv_hat, npv_hat)), "Fill in all four."

# Analytical values from Meeting 2, for comparison
ppv_true = SENS * PREV / (SENS * PREV + (1 - SPEC) * (1 - PREV))
npv_true = SPEC * (1 - PREV) / (SPEC * (1 - PREV) + (1 - SENS) * PREV)

print(f"{'':<14}{'simulated':>12}{'expected':>12}")
for name, got, want in [("sensitivity", sens_hat, SENS), ("specificity", spec_hat, SPEC),
                        ("PPV", ppv_hat, ppv_true),      ("NPV", npv_hat, npv_true)]:
    print(f"{name:<14}{got:>12.4f}{want:>12.4f}")

assert abs(sens_hat - SENS) < 0.02, "Simulated sensitivity is far from the target."
assert abs(spec_hat - SPEC) < 0.02, "Simulated specificity is far from the target."
assert abs(ppv_hat - ppv_true) < 0.01, "Simulated PPV is far from the analytical value."
print("\nAll four match the analytical values.")

The simulated PPV is near 0.045, the same answer as Meeting 2, obtained without using Bayes' rule.

The simulated values do not match the analytical ones exactly, because the simulation draws a finite
sample. The size of the disagreement depends on how many women each quantity is computed from:
specificity is estimated from roughly 199,000 healthy women and agrees closely, while sensitivity is
estimated from roughly 1,100 diseased women and is visibly further off. Question 2 in section 7
returns to this.

Sensitivity and specificity came back at their input values, because they were inputs. PPV and NPV
were not specified anywhere in the simulation — they emerged from the interaction of the test with
the prevalence.

---
## 6 — Sweeping prevalence

Section 5 used one prevalence. Repeating the simulation across a range of prevalences reproduces the
PPV curve from Meeting 2.

> A **sweep** means running the same procedure repeatedly while varying one input, and collecting the
> output each time.

Write a function that runs the whole simulation for a given prevalence and returns the PPV.

In [ ]:
def simulate_ppv(prev, n=50_000, seed=0):
    """Simulate n screened women at the given prevalence and return the observed PPV."""
    rng = np.random.default_rng(seed)
    # --- TODO: repeat sections 3 and 4 inside this function ---
    # The sample size here is `n`, the function's own argument — not the global N.
    # Only TP and FP are needed, since PPV = TP / (TP + FP).

    disease    = None     # who has the disease, at this prevalence
    p_positive = None     # each woman's probability of testing positive
    test       = None     # each woman's test result
    TP         = None     # diseased and tested positive
    FP         = None     # healthy  and tested positive

    if TP is None or FP is None:
        return None
    return TP / (TP + FP)


prevalences = np.array([0.001, 0.003, 0.006, 0.01, 0.03, 0.05, 0.10, 0.25, 0.40])
observed = [simulate_ppv(p, seed=i) for i, p in enumerate(prevalences)]

assert all(o is not None for o in observed), "simulate_ppv is still returning None."
observed = np.array(observed, dtype=float)

expected = SENS * prevalences / (SENS * prevalences + (1 - SPEC) * (1 - prevalences))
print(f"{'prevalence':>12}{'simulated':>12}{'analytical':>12}")
for p, o, e in zip(prevalences, observed, expected):
    print(f"{p:>12.3f}{o:>12.3f}{e:>12.3f}")

assert np.allclose(observed, expected, atol=0.03), "Simulated PPVs do not track the analytical curve."
print("\nSimulation matches theory across three orders of magnitude of prevalence.")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12})

grid = np.logspace(-3.2, -0.35, 300)
curve = SENS * grid / (SENS * grid + (1 - SPEC) * (1 - grid))

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.semilogx(grid, 100 * curve, lw=2.5, color="#c1272d", label="analytical")
ax.semilogx(prevalences, 100 * observed, "o", ms=9, color="#1a1a1a",
            label=f"simulated", zorder=5)
ax.set_xlabel("prevalence  $P(D{=}1)$")
ax.set_ylabel("PPV  (%)")
ax.set_title("Simulated and analytical PPV against prevalence")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

---
## 7 — Submitting your work

**Due at 11:59 PM EST on Wednesday, September 9.**

1. `Runtime ▸ Restart session and run all`. Every cell must run without error, and every check must
   pass.
2. `File ▸ Download ▸ Download .ipynb`.
3. Rename the file to `YOUR-GITHUB-USERNAME_meeting03.ipynb` and move it into the `checkins/` folder
   of your clone.
4. Commit in GitHub Desktop, push, and open a pull request.

The procedure is the same as Meeting 1. If you have forgotten a step, the instructions are in
`notebooks/meeting01_course_launch_and_tools.ipynb`.

### Two short questions

Answer in the cell below before you submit.

In [ ]:
# One or two sentences each.

# 1. Section 5 recovered sensitivity and specificity at their input values, but PPV was never
#    supplied to the simulation. Where did the PPV come from?
Q1 = ""

# 2. Section 3 used N = 200,000 women. If you had used N = 200, which of the four quantities
#    would be least reliable, and why?
Q2 = ""

for name, ans in [("Q1", Q1), ("Q2", Q2)]:
    assert len(ans.strip()) >= 30, f"{name} looks empty or very short."
print("Answers recorded.")